In [23]:
import sys
!{sys.executable} -m pip install -r requirements.txt

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.


In [24]:
import torch
import numpy as np
import pandas as pd
import cv2
import shapely

print("All packages loaded successfully")
print("Torch version:", torch.__version__)

All packages loaded successfully
Torch version: 2.8.0


In [18]:
import sys
!{sys.executable} model/beta_tcvae_pretrain.py

Loading data...
Training samples: 160276
Using device: mps
Starting fixed beta TCVAE pretraining...
Epoch 1/15: 100%|█| 5009/5009 [04:49<00:00, 17.31it/s, kl=66.76, loss=520.35, re
Epoch 01 | Loss: 769.98 | Recon: 685.21 | KL: 49.29 | TC: 35.4747 | Time: 4.82 min
Epoch 2/15: 100%|█| 5009/5009 [07:00<00:00, 11.91it/s, kl=60.79, loss=382.39, re
Epoch 02 | Loss: 432.16 | Recon: 315.30 | KL: 63.87 | TC: 52.9838 | Time: 7.01 min
Epoch 3/15: 100%|█| 5009/5009 [07:42<00:00, 10.82it/s, kl=68.69, loss=433.18, re
Epoch 03 | Loss: 396.75 | Recon: 272.20 | KL: 67.54 | TC: 57.0166 | Time: 7.71 min
Epoch 4/15: 100%|█| 5009/5009 [07:22<00:00, 11.31it/s, kl=67.62, loss=371.68, re
Epoch 04 | Loss: 384.92 | Recon: 257.86 | KL: 68.68 | TC: 58.3765 | Time: 7.38 min
Epoch 5/15: 100%|█| 5009/5009 [07:39<00:00, 10.90it/s, kl=74.15, loss=395.68, re
Epoch 05 | Loss: 377.80 | Recon: 249.78 | KL: 69.11 | TC: 58.9165 | Time: 7.66 min
Epoch 6/15: 100%|█| 5009/5009 [07:47<00:00, 10.71it/s, kl=55.08, loss=267.58, re

In [20]:
import pandas as pd
from pathlib import Path

history_path = Path.home() / "Desktop" / "OOD_training_outputs" / "beta_tcvae" / "beta_tcvae_training_history.csv"

hist = pd.read_csv(history_path)
hist

FileNotFoundError: [Errno 2] No such file or directory: '/Users/paolo/Desktop/OOD_training_outputs/beta_tcvae/beta_tcvae_training_history.csv'

In [22]:
import sys
!{sys.executable} model/OOD_beta_tcvae_classifier.py

Loading data...

Available splits:
split
OOD_train    160276
OOD_test      54329
OOD_hold      52175
Name: count, dtype: int64

Split sizes:
Train: 160276
Test: 54329
Hold: 52175

Train label distribution:
damage_label
no-damage       112434
major-damage     19519
destroyed        15167
minor-damage     13156
Name: count, dtype: int64

Test label distribution:
damage_label
no-damage       32574
minor-damage    12713
destroyed        6022
major-damage     3020
Name: count, dtype: int64

Hold label distribution:
damage_label
no-damage       51815
minor-damage      247
major-damage       77
destroyed          36
Name: count, dtype: int64

Using device: mps

Loading pretrained beta TCVAE encoder...

Fine tuning encoder parameters.

Using class weights: [0.35637796 3.0456827  2.0528204  2.641854  ]

Using cross entropy loss

Starting fixed beta TCVAE classifier training...
Epoch 1/10: 100%|██████████████| 5009/5009 [02:18<00:00, 36.29it/s, loss=0.5909]
Epoch 01 | Train Loss: 1.0582 | Test L

In [17]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix
from pathlib import Path

# =========================
# Configuration
# =========================

split = "hold"   

labels = ["no-damage", "minor-damage", "major-damage", "destroyed"]

PROJECT_ROOT = Path.cwd()

candidate_paths = [
    PROJECT_ROOT / "OOD_training_outputs" / "beta_tcvae_classifier",
    Path.home() / "Desktop" / "OOD_training_outputs" / "beta_tcvae_classifier",
]

BASE_PATH = None
y_true_path = None
y_pred_path = None

# =========================
# Search logic (robust)
# =========================

for path in candidate_paths:
    
    # Try standard naming
    standard_true = path / f"beta_tcvae_{split}_targets.npy"
    standard_pred = path / f"beta_tcvae_{split}_preds.npy"
    
    # Try OOD naming
    ood_true = path / f"OOD_beta_tcvae__{split}_targets.npy"
    ood_pred = path / f"OOD_beta_tcvae__{split}_preds.npy"
    
    if standard_true.exists() and standard_pred.exists():
        BASE_PATH = path
        y_true_path = standard_true
        y_pred_path = standard_pred
        print("Using standard split naming")
        break
        
    elif ood_true.exists() and ood_pred.exists():
        BASE_PATH = path
        y_true_path = ood_true
        y_pred_path = ood_pred
        print("Using OOD split naming")
        break

# =========================
# Fail clearly if not found
# =========================

if BASE_PATH is None:
    print("Files not found. Available .npy files:\n")
    for path in candidate_paths:
        if path.exists():
            print(f"\n{path}:")
            for f in path.glob("*.npy"):
                print("  ", f.name)
    
    raise FileNotFoundError("Could not find matching prediction files.")

print(f"\nUsing files:\n{y_true_path}\n{y_pred_path}")

# =========================
# Load predictions
# =========================

y_true = np.load(y_true_path)
y_pred = np.load(y_pred_path)

# =========================
# Confusion matrices
# =========================

cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(7, 6))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title(f"{split.upper()} Confusion Matrix (Counts)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

plt.figure(figsize=(7, 6))
sns.heatmap(cm_norm, annot=True, fmt=".2f", cmap="Blues",
            xticklabels=labels, yticklabels=labels)
plt.title(f"{split.upper()} Confusion Matrix (Normalized)")
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

Files not found. Available .npy files:



FileNotFoundError: Could not find matching prediction files.